# Set Start to End Date

Format: YYYY-MM-DD

In [ ]:
i = 0
while i == 0:
    user_input = input("Retrieve Full x_post_unique Table? Answer:(Y or N)")
    if user_input == 'N' or user_input == 'n':
        start_date = input("Type Start Date in YYYY-MM-DD format")
        end_date = input("Type END Date in YYYY-MM-DD format")
        QUERY = f"SELECT * FROM x_post_unique WHERE created_at >= '{start_date}' AND created_at < '{end_date}';"
        i = 1
    elif user_input == 'Y' or user_input == 'y':
        QUERY = f"SELECT * FROM x_post_unique;"
        i = 1
    else:
        print('Incorrect Input')
    

In [ ]:
QUERY

# Importing Social Media Data

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from h5n1.db import get_engine

In [ ]:
df = pd.read_sql(QUERY, get_engine())

In [ ]:
print(f'The start date in the dataset: {df['created_at'].min()}')
print(f'The end date in the dataset: {df['created_at'].max()}')

In [ ]:
df

# Clean Text Data
replace \n with space 

In [ ]:
new_text = []
for index, row in df.iterrows():
    new_text.append(df.iloc[index]['post_text'].replace('\n', ' '))
df['cleaned_post_text'] = new_text

In [ ]:
english_text = df.loc[df['lang']=='en']
spanish_text = df.loc[df['lang']=='es']
france_text = df.loc[df['lang']=='fr']

# Sentiment Calculation

Accodrind to this paper: https://arxiv.org/pdf/2405.02887

We can do either of those ways. The paper did suggest that getting sentiment directly from xlm roberta without passing through translation is working a bit better.  

XLM-RoBERTa --> Multi-language Sentiment

OPUS-MT --> Translation --> XLM-RoBERTa --> Sentiment

## XLM-RoBERTa to Sentiment

https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment

* How the model's score works?

Labels: The model has three classes: Negative, Neutral, and Positive. In the model’s config they map to IDs 0 = Negative, 1 = Neutral, 2 = Positive. Score: The score is a probability between 0 and 1, produced by applying a softmax to the model’s three raw outputs. The three probabilities always add up to 1.

The score is the model’s confidence, not how strong the sentiment is. A score of 0.66 means the model is fairly but not very sure the text is positive. It doesn’t mean the text is “66% positive.” If you want one number on a −1 to +1 scale, a common workaround is P(Positive) − P(Negative).

In [ ]:
from transformers import pipeline

In [ ]:
# three scores model
model_path = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
sentiment_task = pipeline("sentiment-analysis", model=model_path, top_k=None)

In [ ]:
# example for spanish
print(spanish_text.iloc[0]['cleaned_post_text'])
sentiment_task(spanish_text.iloc[0]['cleaned_post_text'])

They return all the probabilities on all three classes

In [ ]:
#one score model
model_path = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
sentiment_task = pipeline("sentiment-analysis", model=model_path)

In [ ]:
# example for English
print(english_text.iloc[0]['cleaned_post_text'])
sentiment_task(english_text.iloc[0]['cleaned_post_text'])

Default: return only the class that is highest in probability among three and its probability

In [ ]:
df.iloc[0]['cleaned_post_text']

In [ ]:
texts = df['cleaned_post_text'].fillna('').astype(str).tolist()

In [ ]:
results = sentiment_task(texts[:1000], batch_size=32, truncation=True, max_length=512)

In [ ]:
df_1000 = df.iloc[:1000]
df_1000['label'] = [r['label'] for r in results]
df_1000['score'] = [r['score'] for r in results]